# CallGuard AI - Notebook 06: Caller Type Classification Experiment (AI vs Human vs Robocall)

### Objective
Differentiate between human speakers, conversational AI bots, and automated robocalls.

> **Disclaimer**: Acoustic and linguistic AI voice detection is probabilistic. State-of-the-art TTS models closely mimic natural prosody. Detection should be treated as a risk signal rather than definitive proof.

In [ ]:
# Cell 2: Install dependencies & import libraries
!pip install -q scikit-learn pandas matplotlib seaborn joblib

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

print("Caller type classification libraries loaded.")

In [ ]:
# Cell 3: Load caller type labeled data
data_path = Path("ml/datasets/callguard/processed/cleaned_full.jsonl")
if not data_path.exists():
    data_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

df = pd.read_json(data_path, lines=True)
print("Caller type distribution:")
print(df["caller_type"].value_counts())

In [ ]:
# Cell 4: Feature engineering (linguistic features & turn patterns)
from ml.scripts.feature_engineering import extract_linguistic_features

ling_df = extract_linguistic_features(df["full_transcript"])

# Extract turn pattern features if nested turn list is present
def extract_turn_features(record):
    turns = record.get("conversation", [])
    caller_turns = [t for t in turns if t.get("speaker") == "caller"]
    turn_count = len(turns)
    caller_turn_count = len(caller_turns)
    avg_turn_len = np.mean([len(t.get("text", "").split()) for t in caller_turns]) if caller_turns else 0.0
    return {
        "turn_count": turn_count,
        "caller_turn_count": caller_turn_count,
        "avg_caller_turn_words": float(avg_turn_len)
    }

turn_features_df = pd.DataFrame([extract_turn_features(r) for r in df.to_dict(orient="records")])
all_features_df = pd.concat([ling_df, turn_features_df], axis=1)

print("Engineered caller type features:")
display(all_features_df.head(3))

In [ ]:
# Cell 5: Train classifier
X = all_features_df.values
y = df["caller_type"].values

classes = sorted(list(set(y)))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

caller_rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
caller_rf.fit(X_train, y_train)
print("Caller type Random Forest classifier trained.")

In [ ]:
# Cell 6: Evaluate
y_pred = caller_rf.predict(X_test)

print("=== Caller Type Classification Report ===")
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=classes, yticklabels=classes)
plt.title("Caller Type Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Caller Type")
plt.ylabel("True Caller Type")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: ROC-AUC analysis
y_test_bin = label_binarize(y_test, classes=classes)
y_probs = caller_rf.predict_proba(X_test)

plt.figure(figsize=(9, 6))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Caller Type Multi-Class One-vs-Rest ROC Curves", fontsize=13, fontweight="bold")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# Cell 8: Error analysis — what causes misclassification?
errors = np.where(y_test != y_pred)[0]
print(f"Total misclassifications: {len(errors)} / {len(y_test)}")

if len(errors) > 0:
    for idx in errors[:3]:
        print(f"True: {y_test[idx]} | Predicted: {y_pred[idx]}")
        print(f"Features: {all_features_df.iloc[idx].to_dict()}\n")
else:
    print("No errors detected on the test set.")

# Cell 9: Limitations — voice synthesis, noise, and adversarial callers

### Acoustic & NLP Limitations:
1. **Low-bitrate Telephony Codecs**: AMR-NB (narrowband) filters out frequencies above 3.4 kHz, stripping subtle acoustic artifacts produced by vocoders.
2. **Next-Generation TTS**: Conversational AI engines (e.g., ElevenLabs v3, OpenAI Realtime API) introduce realistic disfluencies ("um", breathing pauses, self-corrections) that confuse purely text-based heuristics.
3. **Multi-Modal Defense**: In production, CallGuard combines transcript structural markers with WebRTC latency timing and audio spectral analysis.